[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C48_Cloud_Deployment_Course/05_cloud_scheduling_cost/05_cloud_scheduling_cost.ipynb)

# 05 · 云平台、集群调度与成本（spot 期望成本、backfill 调度器、$/1M token 分解）

目标：把 **spot 盈亏平衡 → 基线/弹性混合容量 → Slurm backfill 调度 → 单位经济分解** 从零实现，
每个模型都**对拍**蒙特卡洛仿真、每个决策都**算一笔账**。

路线：三件套计价 → spot 期望成本与盈亏点 → 相关性中断的风险 → backfill 调度器 → $/1M token 四因子分解 → ✏️ 练习 → 📖 答案 → 🧪 全课汇总胶囊。

> 心智模型：**成本 = Σ 单价 × 用量**。前四个模块压用量，本模块换单价。
> 两条路都要走，但**先压用量**——便宜单价 × 巨大用量，仍是巨大账单。

## 1 · 三件套的计价不对称

计算按**时间**、存储按**容量×时间+请求数**、网络按**字节且只收出方向**。
这个不对称是所有云成本优化的起点。先把它量化。

In [ ]:
import math, random, heapq
from dataclasses import dataclass, field

# 公开量级（可改成你自己的报价）
PRICE = {
    'gpu_node_hour':   32.00,   # 8×H100 按需
    'storage_gb_month': 0.023,  # 标准对象存储
    'get_per_1k':       0.0004, # GET 请求
    'egress_gb':        0.09,   # 出公网/跨区
    'cross_az_gb':      0.01,   # 跨可用区
}

def monthly_bill(nodes, weights_gb, n_model_versions, pulls_per_month, cross_az_gb_per_month):
    compute = nodes * PRICE['gpu_node_hour'] * 24 * 30
    storage = weights_gb * n_model_versions * PRICE['storage_gb_month']
    egress  = pulls_per_month * weights_gb * PRICE['egress_gb']
    az      = cross_az_gb_per_month * PRICE['cross_az_gb']
    return {'计算': compute, '存储': storage, '权重拉取': egress, '跨AZ': az,
            '合计': compute + storage + egress + az}

bill = monthly_bill(nodes=7, weights_gb=140, n_model_versions=5,
                    pulls_per_month=60, cross_az_gb_per_month=20000)
for k, v in bill.items():
    share = v / bill['合计']
    print(f'{k:>10s}: ${v:>12,.2f}  {share:>6.1%}')

net = bill['权重拉取'] + bill['跨AZ']
assert bill['计算'] > 0.8 * bill['合计'], '计算应是大头'
assert net > bill['存储'] * 30, '网络费远大于存储费 —— 存储便宜，搬运贵'
print(f'\n✅ 存储只要 ${bill["存储"]:,.0f}，但「把它搬来搬去」要 ${net:,.0f}（{net/bill["存储"]:.0f} 倍）')
print('   记住这条不对称：**存字节便宜，搬字节贵**。')

## 2 · spot 经济学：盈亏平衡的中断率

$$\mathbb{E}[C_{spot}] = p_{spot} + h \cdot c_{int}, \qquad h^* = \frac{p_{od} - p_{spot}}{c_{int}}$$

`h*` 是**盈亏平衡中断率**：实际中断率低于它，spot 划算；高于它，spot 反而更贵。

In [ ]:
def spot_breakeven(p_od, p_spot, c_interrupt):
    '''返回每小时盈亏平衡中断次数。'''
    return (p_od - p_spot) / c_interrupt

def spot_expected_cost(p_spot, hazard_per_hour, c_interrupt):
    return p_spot + hazard_per_hour * c_interrupt

P_OD, P_SPOT = 32.0, 10.0        # 按需 $32/h，spot $10/h（约 -69%）

workloads = [
    ('离线批推理(可断点续)',  0.5),    # 一次中断只丢几分钟工作
    ('预训练(30min ckpt)',   16.0),    # 丢半小时 × 全体 rank
    ('在线推理(无热备)',    600.0),    # 容量缺口 + 请求失败 + 3 分钟冷启动的 SLO 损失
]
print(f"{'工作负载':<24s} {'c_int($)':>9s} {'盈亏中断率/h':>13s} {'实际0.05/h时期望$':>18s}")
for name, c_int in workloads:
    h_star = spot_breakeven(P_OD, P_SPOT, c_int)
    exp = spot_expected_cost(P_SPOT, 0.05, c_int)
    verdict = '✅ 划算' if exp < P_OD else '❌ 更贵'
    print(f'{name:<24s} {c_int:>9.1f} {h_star:>13.3f} {exp:>15.2f} {verdict}')

assert spot_breakeven(P_OD, P_SPOT, 0.5) > 40, '批处理能容忍极高中断率'
assert spot_expected_cost(P_SPOT, 0.05, 600.0) > P_OD, '在线推理裸用 spot 反而更贵'
print('\n✅ 同一个折扣，对不同工作负载的结论完全相反 —— 决定因素是 c_int，不是折扣力度')

### 对拍：蒙特卡洛仿真 vs 解析期望

In [ ]:
def simulate_spot(hours, p_spot, hazard, c_int, seed=0):
    rng = random.Random(seed)
    total = 0.0
    for _ in range(hours):
        total += p_spot
        if rng.random() < hazard:
            total += c_int
    return total / hours

for hz, c_int in [(0.02, 16.0), (0.05, 16.0), (0.10, 0.5)]:
    sim = simulate_spot(200000, P_SPOT, hz, c_int)
    ana = spot_expected_cost(P_SPOT, hz, c_int)
    rel = abs(sim - ana) / ana
    print(f'hazard={hz:.2f} c_int=${c_int:>5.1f}: 仿真 ${sim:>6.3f}/h | 解析 ${ana:>6.3f}/h | 误差 {rel:.2%}')
    assert rel < 0.02, '解析式应与蒙特卡洛吻合'
print('✅ 对拍通过：期望成本公式正确')

### 相关性中断：spot 不是独立事件

现实里区域容量紧张时，**所有 spot 可能在几分钟内同时消失**。
设计时必须问：全部 spot 消失后，剩余容量够不够撑住 P50 流量？

In [ ]:
def survive_spot_wipeout(baseline_replicas, spot_replicas, mu, p50_qps, target_rho=0.9):
    '''全部 spot 被收回后，仅靠基线能否撑住 P50 流量。'''
    capacity = baseline_replicas * mu * target_rho
    return capacity >= p50_qps, capacity

MU, P50_QPS, PEAK_QPS = 2.5, 60.0, 200.0
designs = [
    ('全 spot',        0,  ceil_ := math.ceil(PEAK_QPS / (MU * 0.7))),
    ('全按需',         ceil_, 0),
    ('基线覆盖P50+spot弹性', math.ceil(P50_QPS / (MU * 0.7)), ceil_ - math.ceil(P50_QPS / (MU * 0.7))),
]
print(f"{'设计':<24s} {'基线':>5s} {'spot':>5s} {'月成本$':>11s} {'spot全灭后':>12s}")
for name, base, spot in designs:
    cost = (base * P_OD + spot * P_SPOT) / 4 * 24 * 30    # 每节点 4 副本
    ok, cap = survive_spot_wipeout(base, spot, MU, P50_QPS)
    print(f'{name:<24s} {base:>5d} {spot:>5d} {cost:>11,.0f} '
          f'{"✅ 撑得住" if ok else "❌ 服务不可用":>12s}')

all_spot_ok, _ = survive_spot_wipeout(0, ceil_, MU, P50_QPS)
mixed_ok, _ = survive_spot_wipeout(math.ceil(P50_QPS/(MU*0.7)), 0, MU, P50_QPS)
assert not all_spot_ok, '全 spot 在集体收回时服务完全不可用'
assert mixed_ok, '基线覆盖 P50 的混合设计能扛住 spot 全灭'
print('\n✅ 标准做法：**基线（按需/预留）覆盖 P50 + 弹性（spot）覆盖峰值**')
print('   最坏情况从「服务不可用」退化成「性能降级」—— 这才是可接受的失效模式。')

## 3 · Slurm 的核心：backfill 调度器

Slurm 比 K8s 默认调度器更懂「有明确时长的作业」，因此能做 **backfill**：
在等大作业攒资源的空隙里，插入那些「声明时长足够短、能在空隙内跑完」的小作业。

这解释了那条实用经验：**`--time` 填得越准（越短），排队等得越少。**

In [ ]:
@dataclass
class Job:
    jid: int
    nodes: int
    walltime: int      # 声明时长（秒）
    submit: int = 0
    prio: int = 0

def fcfs_schedule(jobs, total_nodes):
    '''纯先来先服务：队首作业攒不够资源就全队阻塞。'''
    t, free, running, done = 0, total_nodes, [], []
    q = sorted(jobs, key=lambda j: (-j.prio, j.submit, j.jid))
    while q or running:
        while q and q[0].nodes <= free:
            j = q.pop(0); free -= j.nodes
            heapq.heappush(running, (t + j.walltime, j.jid, j.nodes))
        if not running:
            break
        t, jid, n = heapq.heappop(running)
        free += n; done.append((jid, t))
    return done, t

def backfill_schedule(jobs, total_nodes):
    '''EASY backfill：队首作业保留预约，后面的小作业若能在预约前跑完就插队。'''
    t, free, running, done = 0, total_nodes, [], []
    q = sorted(jobs, key=lambda j: (-j.prio, j.submit, j.jid))
    while q or running:
        # 1) 尽量启动队首
        while q and q[0].nodes <= free:
            j = q.pop(0); free -= j.nodes
            heapq.heappush(running, (t + j.walltime, j.jid, j.nodes))
        # 2) 为队首算「预约开始时刻」，再回填能在此之前结束的小作业
        if q:
            need, tmp, f2, res_t = q[0].nodes, list(running), free, t
            while f2 < need and tmp:
                et, _, n = heapq.heappop(tmp); f2 += n; res_t = et
            i = 1
            while i < len(q):
                j = q[i]
                if j.nodes <= free and t + j.walltime <= res_t:
                    q.pop(i); free -= j.nodes
                    heapq.heappush(running, (t + j.walltime, j.jid, j.nodes))
                else:
                    i += 1
        if not running:
            break
        t, jid, n = heapq.heappop(running)
        free += n; done.append((jid, t))
    return done, t

TOTAL_NODES = 16
# 关键场景：一个长作业占住半个集群 -> 队首的整集群作业只能等 -> 空出的 8 节点该不该闲着？
jobs = [Job(0, nodes=8,  walltime=3600),        # 已在跑，占 8 节点 1 小时
        Job(1, nodes=16, walltime=1800)] + \
       [Job(i, nodes=2, walltime=600) for i in range(2, 10)]     # 8 个 10 分钟小作业

d_fcfs, mk_fcfs = fcfs_schedule(jobs, TOTAL_NODES)
d_bf,   mk_bf   = backfill_schedule(jobs, TOTAL_NODES)
print(f'FCFS     完成时刻(makespan): {mk_fcfs/60:>6.1f} 分钟')
print(f'Backfill 完成时刻(makespan): {mk_bf/60:>6.1f} 分钟')
avg_fcfs = sum(t for _, t in d_fcfs) / len(d_fcfs)
avg_bf   = sum(t for _, t in d_bf) / len(d_bf)
print(f'平均完成时间: FCFS {avg_fcfs/60:.1f} 分钟 | Backfill {avg_bf/60:.1f} 分钟')
assert mk_bf <= mk_fcfs, 'backfill 不应变差'
assert avg_bf < avg_fcfs, 'backfill 应显著改善平均完成时间'
print(f'\n✅ backfill 把平均完成时间改善 {(1-avg_bf/avg_fcfs):.0%} —— 空隙被小作业填满了')

### `--time` 填得越短，越容易被回填

In [ ]:
def time_to_start(declared_walltime, total_nodes=16, probe_jid=2):
    '''同一个作业（真实只跑 10 分钟），声明不同 --time 时的实际启动时刻。'''
    js = [Job(0, nodes=8,  walltime=3600),
          Job(1, nodes=16, walltime=1800),
          Job(probe_jid, nodes=2, walltime=declared_walltime)]
    t, free, running, start = 0, total_nodes, [], None
    q = sorted(js, key=lambda j: j.jid)
    while q or running:
        while q and q[0].nodes <= free:
            j = q.pop(0); free -= j.nodes
            if j.jid == probe_jid and start is None: start = t
            heapq.heappush(running, (t + j.walltime, j.jid, j.nodes))
        if q:
            need, tmp, f2, res_t = q[0].nodes, list(running), free, t
            while f2 < need and tmp:
                et, _, n = heapq.heappop(tmp); f2 += n; res_t = et
            i = 1
            while i < len(q):
                j = q[i]
                if j.nodes <= free and t + j.walltime <= res_t:
                    q.pop(i); free -= j.nodes
                    if j.jid == probe_jid and start is None: start = t
                    heapq.heappush(running, (t + j.walltime, j.jid, j.nodes))
                else:
                    i += 1
        if not running: break
        t, jid, n = heapq.heappop(running); free += n
    return start

print(f"{'声明 --time':>14s} {'实际启动时刻':>14s}")
for wt in [600, 1800, 3600, 7200]:
    s = time_to_start(wt)
    print(f'{wt//60:>11d} 分 {s/60 if s is not None else -1:>13.1f} 分')
assert time_to_start(600) < time_to_start(7200), '声明时长越短，越早被回填调度'
print('\n✅ 同一个作业（真实只跑 10 分钟），声明 10 分钟 vs 120 分钟，排队差了一小时。')
print('   实用建议：`--time` 填真实需要的 1.2~1.5 倍，配 checkpoint + --requeue 兜底。')

## 4 · 单位经济：$ / 1M token 的四因子分解

$$\text{\$/1M tok} = \frac{p_{node}}{T_{peak} \times u \times \eta \times 3600 / 10^6}$$

这个分解告诉你**优化该往哪儿使劲**。

In [ ]:
def cost_per_1m_tokens(p_node_hour, t_peak_tok_s, utilization, packing):
    eff_tokens_per_hour = t_peak_tok_s * utilization * packing * 3600
    return p_node_hour / (eff_tokens_per_hour / 1e6)

configs = [
    ('最差  (无扩缩/碎片严重)', 32.0, 4000, 0.25, 0.50),
    ('典型  (基本配置)',        32.0, 4000, 0.45, 0.70),
    ('优化  (本课全套)',        32.0, 5500, 0.65, 0.85),
    ('优化+spot',               10.0, 5500, 0.65, 0.85),
]
print(f"{'配置':<26s} {'$/节点h':>9s} {'吞吐':>6s} {'利用率':>7s} {'装箱':>6s} {'$/1M tok':>10s}")
costs = []
for name, p, t, u, e in configs:
    c = cost_per_1m_tokens(p, t, u, e); costs.append(c)
    print(f'{name:<26s} {p:>9.2f} {t:>6d} {u:>7.2f} {e:>6.2f} {c:>10.3f}')

assert costs == sorted(costs, reverse=True), '成本应逐行下降'
print(f'\n最差 -> 最优 = {costs[0]/costs[-1]:.1f} 倍差距，**没有一项来自「换更好的 GPU」**')

# 敏感度分析：各因子单独改善 30% 的收益
base = cost_per_1m_tokens(32.0, 4000, 0.45, 0.70)
print(f'\n基线 ${base:.3f}/1M tok。各因子单独改善 30% 的收益:')
for label, kw in [('吞吐 +30%', dict(t_peak_tok_s=5200)),
                  ('利用率 +30%', dict(utilization=0.585)),
                  ('装箱率 +30%', dict(packing=0.91)),
                  ('单价 -30%', dict(p_node_hour=22.4))]:
    args = dict(p_node_hour=32.0, t_peak_tok_s=4000, utilization=0.45, packing=0.70)
    args.update(kw)
    c = cost_per_1m_tokens(**args)
    print(f'  {label:<12s} -> ${c:.3f} (省 {(1-c/base):.0%})')
print('\n✅ 四个因子的边际收益相同（都是 ~23%）—— 但**改善难度天差地别**：')
print('   榨 30% 吞吐要 kernel 工程；提 30% 装箱率只要改一行调度策略。**先摘低垂的果子。**')

## ✏️ 练习 1：混合容量的最优 spot 比例

实现 `optimal_spot_ratio(peak_r, p50_r, p_od, p_spot, wipeout_prob, downtime_cost_per_hour)`：
在「基线必须能撑住 P50」的约束下，返回 `(baseline_replicas, spot_replicas, expected_hourly_cost)`。
- `baseline = p50_r`（刚好覆盖 P50），`spot = peak_r - p50_r`
- 期望小时成本 = `baseline*p_od + spot*p_spot + wipeout_prob * downtime_cost_per_hour`
  （spot 全灭时只是降级不是宕机，所以这里的 downtime cost 按「峰值时段容量不足」计）

In [ ]:
def optimal_spot_ratio(peak_r, p50_r, p_od, p_spot, wipeout_prob, downtime_cost_per_hour):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
b, s, c = optimal_spot_ratio(peak_r=115, p50_r=35, p_od=8.0, p_spot=2.5,
                             wipeout_prob=0.01, downtime_cost_per_hour=500.0)
assert b == 35 and s == 80, f'基线覆盖 P50，其余用 spot，得到 ({b}, {s})'
expected = 35*8.0 + 80*2.5 + 0.01*500.0
assert abs(c - expected) < 1e-9
# 对比全按需
all_od = 115 * 8.0
print(f'混合: 基线 {b} + spot {s} = ${c:.2f}/h')
print(f'全按需: ${all_od:.2f}/h')
assert c < all_od * 0.7, '混合方案应至少省 30%'
print(f'✅ 练习 1 通过：省 {(1-c/all_od):.0%}，且最坏情况只是降级不是宕机')

## ✏️ 练习 2：backfill 可行性判定

实现 `can_backfill(job_nodes, job_walltime, free_nodes, now, reservation_time)`：
判断一个作业能否被回填——需要同时满足
①`job_nodes <= free_nodes`；②`now + job_walltime <= reservation_time`（不能推迟队首的预约）。

In [ ]:
def can_backfill(job_nodes, job_walltime, free_nodes, now, reservation_time):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert can_backfill(2, 600, free_nodes=4, now=0, reservation_time=3600) is True
assert can_backfill(8, 600, free_nodes=4, now=0, reservation_time=3600) is False, '节点不够'
assert can_backfill(2, 7200, free_nodes=4, now=0, reservation_time=3600) is False, '会推迟队首预约'
assert can_backfill(2, 3600, free_nodes=4, now=0, reservation_time=3600) is True, '刚好卡住是允许的'
# 单调性：声明时长越短越容易回填
ok = [can_backfill(2, wt, 4, 0, 3600) for wt in [600, 1800, 3600, 5400]]
assert ok == [True, True, True, False]
print('✅ 练习 2 通过：这两个条件就是 EASY backfill 的全部判据')

## ✏️ 练习 3：成本归因

实现 `attribute_cost(total_node_hours, tenant_tokens, node_hourly)`：
`tenant_tokens` 是 `{租户: token 数}`。按 token 比例分摊总计算成本。
返回 `{租户: 分摊金额}`，并保证总和等于总成本（浮点误差 < 1e-6）。

In [ ]:
def attribute_cost(total_node_hours, tenant_tokens, node_hourly):
    # TODO: total = total_node_hours * node_hourly；按 token 占比分摊
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
alloc = attribute_cost(720, {'chat': 8_000_000, 'batch': 2_000_000, 'internal': 500_000}, 32.0)
total = 720 * 32.0
print({k: round(v, 2) for k, v in alloc.items()})
assert abs(sum(alloc.values()) - total) < 1e-6, '分摊总和必须等于总成本'
assert alloc['chat'] > alloc['batch'] > alloc['internal'], '按用量排序'
assert abs(alloc['chat'] / total - 8/10.5) < 1e-9
# 边界：无用量时不应崩
empty = attribute_cost(720, {}, 32.0)
assert empty == {} or abs(sum(empty.values())) < 1e-9
print('✅ 练习 3 通过：按 token 分摊是最常用的归因方式（但对共享批处理并不完全公平）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def optimal_spot_ratio(peak_r, p50_r, p_od, p_spot, wipeout_prob, downtime_cost_per_hour):
    baseline = p50_r
    spot = max(0, peak_r - p50_r)
    cost = baseline * p_od + spot * p_spot + wipeout_prob * downtime_cost_per_hour
    return baseline, spot, cost

In [ ]:
# 练习 2 参考答案
def can_backfill(job_nodes, job_walltime, free_nodes, now, reservation_time):
    return job_nodes <= free_nodes and now + job_walltime <= reservation_time

In [ ]:
# 练习 3 参考答案
def attribute_cost(total_node_hours, tenant_tokens, node_hourly):
    total = total_node_hours * node_hourly
    s = sum(tenant_tokens.values())
    if s == 0:
        return {k: 0.0 for k in tenant_tokens}
    return {k: total * v / s for k, v in tenant_tokens.items()}

---
## 🧪 真实数据胶囊：把全课五个模块的决策汇成一张账

把模块 01–05 的每个决策，换算成对 `$/1M token` 的贡献。这是本课的最终答卷。

In [ ]:
BASE = dict(p_node_hour=32.0, t_peak_tok_s=4000, utilization=0.30, packing=0.50)
baseline_cost = cost_per_1m_tokens(**BASE)

improvements = [
    ('模块01 多阶段镜像+本地权重缓存 -> 冷启动 180s→45s',   dict(utilization=0.36)),
    ('模块02 容量按 Erlang-C 规划，不再盲目超配',           dict(utilization=0.42)),
    ('模块03 装箱调度 + 跨AZ均分（弃硬反亲和）',            dict(packing=0.78)),
    ('模块04 HPA 按队列深度 + 稳定窗口，超配减少',          dict(utilization=0.58)),
    ('模块05 基线预留 + 弹性 spot',                          dict(p_node_hour=18.0)),
]
cur = dict(BASE)
print(f'起点: ${baseline_cost:.3f} / 1M tok\n')
prev = baseline_cost
for name, delta in improvements:
    cur.update(delta)
    c = cost_per_1m_tokens(**cur)
    print(f'{name}\n    ${prev:.3f} -> ${c:.3f}  (本步省 {(1-c/prev):>4.0%}, 累计省 {(1-c/baseline_cost):>4.0%})\n')
    prev = c

final = cost_per_1m_tokens(**cur)
assert final < baseline_cost / 5, f'全套优化应至少降到 1/5，实际 {baseline_cost/final:.1f}x'
print(f'✅ 累计 {baseline_cost/final:.1f} 倍改善 —— **没有一步来自换更好的 GPU 或改模型**。')
print('   这就是本课的全部论点：部署工程的复合收益，与模型优化同量级，但便宜得多。')

**🧪 胶囊练习**：实现 `payback_months(engineering_days, day_rate, monthly_saving)`：
一项优化花了 `engineering_days` 人天（每人天成本 `day_rate`），每月省 `monthly_saving`。
返回**回本月数**（向上取整）；若 `monthly_saving <= 0` 返回 `None`。

In [ ]:
def payback_months(engineering_days, day_rate, monthly_saving):
    # TODO
    raise NotImplementedError

In [ ]:
# 自测
monthly_before = baseline_cost * 3000    # 假设每月 30 亿 token
monthly_after  = final * 3000
saving = monthly_before - monthly_after
m = payback_months(20, 800, saving)
assert m is not None and m >= 1
assert payback_months(20, 800, 0) is None
assert payback_months(20, 800, -5) is None
print(f'每月 30 亿 token: ${monthly_before:,.0f} -> ${monthly_after:,.0f}，月省 ${saving:,.0f}')
print(f'投入 20 人天 (${20*800:,}) -> 回本 {m} 个月')
assert m <= 2, '这类基础设施优化通常一两个月就回本'
print('✅ 胶囊练习通过：**部署优化几乎总是投入产出比最高的工程**')

In [ ]:
# 📖 胶囊参考答案
def payback_months(engineering_days, day_rate, monthly_saving):
    if monthly_saving <= 0:
        return None
    return math.ceil(engineering_days * day_rate / monthly_saving)

---
## 🔧 旁注：真实系统里这些对应什么

- **三件套计价** → AWS Cost Explorer / GCP Billing 的「按服务 + 按用量类型」双维度切分；第一次做务必两个维度都看。
- **spot 中断处理** → AWS EC2 Spot ITN（`/latest/meta-data/spot/instance-action`）、GCP preemption notice；收到后触发模块 01 的优雅停机。
- **多样化 spot 池** → EC2 Fleet / Karpenter 的多实例类型 + 多 AZ 配置；SkyPilot 做跨云 spot 容错。
- **Slurm backfill** → `SchedulerType=sched/backfill`；`scontrol show job <id>` 的 `Reason` 字段告诉你为什么还在排队。
- **K8s 上的批作业排队** → Kueue（原生 gang scheduling + 配额）、Volcano；不加插件的裸 K8s 跑训练作业利用率通常很差。
- **单位经济归因** → OpenCost / Kubecost 按 namespace/label 分摊；LLM 服务还要在应用层记 `model_version` + `tenant` 的 token 计数。

至此，五个模块的链路完整：**镜像（01）→ 契约（02）→ 编排（03）→ 发布扩缩（04）→ 云与成本（05）**。

### 小结
- 云只有三件套，且**计价单位不同**：计算按时间、存储按容量、网络按字节且只收出方向。**存字节便宜，搬字节贵。**
- **spot 的取舍由 `c_int` 决定，不是由折扣力度决定**；盈亏平衡中断率 `h* = (p_od − p_spot)/c_int`。
- spot 中断是**相关的**：设计时必须回答「全部 spot 消失后还剩多少容量」。标准做法是**基线（按需/预留）覆盖 P50 + 弹性 spot 覆盖峰值**。
- **K8s Job / Slurm / Ray 是三种世界观**；Slurm 的 backfill 让「`--time` 填得准」直接换来更短的排队。
- **$/1M token = p_node / (T_peak × u × η × 3600/1e6)**：四个因子边际收益相同，但改善难度天差地别。**先摘低垂的果子。**
- 全课汇总：五个模块的决策复合起来能带来一个数量级的成本改善，**没有一项依赖更好的 GPU 或更好的模型**。

🎓 **本课完结。** 你现在有了一条从「进程」到「服务」的完整链路，以及给每个环节算账的能力。
建议的下一站：**C37 MLOps**（生命周期与监控）、**C24 推理服务**（引擎内部机制）、**C39 分布式训练**（多节点工程）。